# Memory

LLMs are stateless

2 Core Design Decision:  
1. How state is stored
2. How state is queried 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer all questions to the best 
        of your ability."""),
    ("placeholder", "{messages}"),
])

model = ChatOpenAI()

chain = prompt | model

# Simple Memory is just to append previous interaction as context to new query
chain.invoke({
    "messages": [
        ("human","""Translate this sentence from English to French: I love 
            programming."""),
        ("ai", "J'adore programmer."),
        ("human", "What did you just say?"),
    ],
})

AIMessage(content='I just said "J\'adore programmer," which means "I love programming" in French.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 63, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Blfgg0Ocb9EgJphcdbGju1tYZDLuN', 'finish_reason': 'stop', 'logprobs': None}, id='run-725f2ab0-1591-4639-b902-ccbd4cdf9e43-0', usage_metadata={'input_tokens': 63, 'output_tokens': 20, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Problem with this simple approach
1. Update Memory after every interaction
2. Need durable database for disk storage
3. Control how many and which message to store
4. Modify message as need be.

## LangGraph

Enables Multi-Actor, Multi-Step, stateful cognitive architecture called graphs

Each graph consists of 3 things:
1. State   
    Current state of output; Imagine it as a flow; Travels to node through edges
2. Node   
    Function that takes state as input and output a updated state
3. Edges    
    Define which node will state visit next.

### State Graph

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    # Messages have the type "list". The `add_messages` 
    # function in the annotation defines how this state should 
    # be updated (in this case, it appends new messages to the 
    # list, rather than replacing the previous messages)
	messages: Annotated[list, add_messages]

builder = StateGraph(State)

### Adding Memory to State Graph

### Modify Chat History

#### -> Trimming Message

#### -> Filtering Message

#### -> Merging Messages